In [ ]:
"""RSNA Knee 数据加载与 DICOM 预处理模块。"""

from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


KAGGLE_INPUT = Path("/kaggle/input")


def find_data_root(root=None):
    """Return the mounted Kaggle dataset containing the RSNA CSV files.

    Pass ``--data-root`` to train.py if more than one mounted dataset contains
    a train.csv.  This keeps the notebook independent of Kaggle dataset slugs.
    """
    if root is not None:
        root = Path(root)
        if (root / "train.csv").is_file():
            return root
        raise FileNotFoundError(f"RSNA data root has no train.csv: {root}")
    candidates = [path.parent for path in KAGGLE_INPUT.rglob("train.csv")
                  if (path.parent / "train_series.csv").is_file()
                  and (path.parent / "test.csv").is_file()
                  and (path.parent / "test_series.csv").is_file()]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "Could not find RSNA data under /kaggle/input. Attach the competition "
            "data, or provide --data-root /kaggle/input/<your-data-dataset>."
        )
    raise RuntimeError(
        "Multiple possible RSNA data roots found: " + ", ".join(map(str, candidates)) +
        ". Provide --data-root explicitly."
    )


DEFAULT_ROOT = None
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = ["Axial_FS", "Axial_nonFS", "Coronal_FS", "Coronal_nonFS", "Sagittal_FS", "Sagittal_nonFS"]
SERIES_FEATURES = ["fluid_sensitive", "fat_suppression", "t1", "pd", "t2", "contrast_unknown", "te", "tr", "pixel_spacing", "slice_spacing", "coverage"]


def _count_dicoms(path):
    return sum(1 for _ in Path(path).glob("*.dcm"))


def _prepare_series_df(series_df, image_root):
    series_df = series_df.copy()
    series_df["SeriesPath"] = [Path(image_root) / str(study) / str(series) for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID)]
    series_df["SeriesSlot"] = series_df.Anatomical_Plane + "_" + np.where(series_df.Fat_Suppression.eq(1), "FS", "nonFS")
    series_df["NumSlices"] = [_count_dicoms(path) for path in series_df.SeriesPath]
    return series_df


def load_metadata(root=DEFAULT_ROOT):
    root = find_data_root(root)
    train_df, test_df = pd.read_csv(root / "train.csv"), pd.read_csv(root / "test.csv")
    train_series_df = _prepare_series_df(pd.read_csv(root / "train_series.csv"), root / "train_series")
    test_series_df = _prepare_series_df(pd.read_csv(root / "test_series.csv"), root / "test_series")
    return train_df, train_series_df, test_df, test_series_df, pd.read_csv(root / "sample_submission.csv")


def get_sorted_dicom_info(series_path):
    """使用患者坐标排序，并把主切片方向统一为患者坐标轴的正方向。"""
    files = list(Path(series_path).glob("*.dcm"))
    headers = [pydicom.dcmread(file, stop_before_pixels=True) for file in files]
    orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=np.float32)
    normal = np.cross(orientation[:3], orientation[3:])
    normal /= np.linalg.norm(normal)
    if normal[np.argmax(np.abs(normal))] < 0:
        normal = -normal
    positions = np.asarray([np.dot(np.asarray(header.ImagePositionPatient, dtype=np.float32), normal) for header in headers])
    order = np.argsort(positions)
    return [files[i] for i in order], positions[order], [headers[i] for i in order]


def sort_dicom_files(series_path):
    return get_sorted_dicom_info(series_path)[0]


def read_dicom(path):
    dicom = pydicom.dcmread(path)
    try:
        image = dicom.pixel_array.astype(np.float32)
    except ValueError as error:
        # 少数文件的头信息写着16位，但未压缩PixelData实际是8位；按真实字节数恢复。
        rows, columns = int(dicom.Rows), int(dicom.Columns)
        frames = int(getattr(dicom, "NumberOfFrames", 1) or 1)
        samples = int(getattr(dicom, "SamplesPerPixel", 1) or 1)
        pixel_count = rows * columns * frames * samples
        raw = dicom.PixelData
        transfer_syntax = getattr(getattr(dicom, "file_meta", None), "TransferSyntaxUID", None)
        uncompressed = transfer_syntax is None or not transfer_syntax.is_compressed
        if not uncompressed or frames != 1 or samples != 1 or len(raw) not in (pixel_count, pixel_count + 1):
            raise ValueError(f"Cannot decode DICOM {path}: {error}") from error
        dtype = np.int8 if int(getattr(dicom, "PixelRepresentation", 0) or 0) else np.uint8
        image = np.frombuffer(raw[:pixel_count], dtype=dtype).reshape(rows, columns).astype(np.float32)
    image = image * float(getattr(dicom, "RescaleSlope", 1)) + float(getattr(dicom, "RescaleIntercept", 0))
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() + image.min() - image
    return image


def _read_sampled_slices(files, indices):
    """读取采样切片；单张损坏时用空间上最近的可读切片替代。"""
    cache, invalid = {}, set()
    images = []
    for index in indices:
        index = int(index)
        candidates = sorted(range(len(files)), key=lambda candidate: abs(candidate - index))
        for candidate in candidates:
            if candidate in cache:
                image = cache[candidate]
                break
            if candidate in invalid:
                continue
            try:
                image = read_dicom(files[candidate])
                cache[candidate] = image
                break
            except (ValueError, OSError, EOFError, InvalidDicomError):
                invalid.add(candidate)
        else:
            raise RuntimeError(f"No decodable DICOM slices in series: {Path(files[0]).parent}")
        images.append(image)
    return np.stack(images)


def _sample_by_position(positions, num_slices):
    """在真实物理覆盖范围内均匀采样，而不是只按切片编号采样。"""
    targets = np.linspace(positions[0], positions[-1], num_slices)
    return np.abs(positions[:, None] - targets[None, :]).argmin(axis=0)


def _contrast_features(te, tr, fat_suppression):
    """依据 EDA 使用的 TE/TR 启发式生成 T1、PD、T2、未知四维标记。"""
    if fat_suppression:
        return [0, 1, 1, 0]
    if te <= 0 or tr <= 0:
        return [0, 0, 0, 1]
    if tr < 1000 and te < 30:
        return [1, 0, 0, 0]
    if te < 60:
        return [0, 1, 0, 0]
    return [0, 0, 1, 0]


def load_series(series_path, num_slices=24, image_size=224, fluid_sensitive=0, fat_suppression=0, return_features=False):
    """空间排序→物理位置采样→DICOM 解码→序列级归一化→等比例缩放。"""
    files, positions, headers = get_sorted_dicom_info(series_path)
    indices = _sample_by_position(positions, num_slices)
    volume = _read_sampled_slices(files, indices)

    foreground = volume[np.isfinite(volume) & (volume != 0)]
    low, high = np.percentile(foreground, [0.5, 99.5])
    volume = np.clip((volume - low) / (high - low + 1e-6), 0, 1).astype(np.float32)
    volume = torch.from_numpy(volume)[:, None]
    height, width = volume.shape[-2:]
    pad_h, pad_w = max(height, width) - height, max(height, width) - width
    volume = F.pad(volume, (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2))
    volume = F.interpolate(volume, size=(image_size, image_size), mode="bilinear", align_corners=False)

    header = headers[len(headers) // 2]
    te, tr = float(getattr(header, "EchoTime", 0) or 0), float(getattr(header, "RepetitionTime", 0) or 0)
    spacing = np.asarray(getattr(header, "PixelSpacing", [0, 0]), dtype=np.float32)
    slice_spacing = float(np.median(np.abs(np.diff(positions)))) if len(positions) > 1 else 0
    coverage = float(abs(positions[-1] - positions[0])) if len(positions) > 1 else 0
    contrast = _contrast_features(te, tr, int(fat_suppression))
    features = torch.tensor([float(fluid_sensitive), float(fat_suppression), *contrast, te / 100, tr / 5000, float(spacing.mean()), slice_spacing / 10, coverage / 200], dtype=torch.float32)
    return (volume, features) if return_features else volume


class KneeDataset(Dataset):
    """Study 级数据集，输出 images [6,S,1,H,W] 和六个槽位的元数据。"""

    def __init__(self, study_df, series_df, num_slices=24, image_size=224):
        self.study_df = study_df.reset_index(drop=True)
        self.num_slices, self.image_size = num_slices, image_size
        self.series_groups = {uid: group.reset_index(drop=True) for uid, group in series_df.groupby("StudyInstanceUID")}

    def __len__(self):
        return len(self.study_df)

    def __getitem__(self, index):
        row, uid = self.study_df.iloc[index], self.study_df.iloc[index].StudyInstanceUID
        series_df = self.series_groups[uid]
        images = torch.zeros(len(SLOTS), self.num_slices, 1, self.image_size, self.image_size)
        slot_mask = torch.zeros(len(SLOTS))
        series_features = torch.zeros(len(SLOTS), len(SERIES_FEATURES))

        for slot_index, slot in enumerate(SLOTS):
            candidates = series_df[series_df.SeriesSlot == slot]
            if len(candidates):
                selected = candidates.sort_values(["Fluid_Sensitive", "NumSlices"], ascending=[False, False]).iloc[0]
                volume, features = load_series(selected.SeriesPath, self.num_slices, self.image_size, selected.Fluid_Sensitive, selected.Fat_Suppression, True)
                images[slot_index], series_features[slot_index], slot_mask[slot_index] = volume, features, 1

        values = pd.to_numeric(row[LABELS], errors="coerce").to_numpy(dtype=np.float32) if set(LABELS).issubset(row.index) else np.full(len(LABELS), np.nan, dtype=np.float32)
        valid = ~np.isnan(values)
        label_weight = np.where(valid, 2 * np.abs(values - 0.5), 0).astype(np.float32)
        return {
            "study_uid": uid,
            "images": images,
            "slot_mask": slot_mask,
            "series_features": series_features,
            "targets": torch.from_numpy(np.nan_to_num(values, nan=0)),
            "label_mask": torch.from_numpy(valid.astype(np.float32)),
            "label_weight": torch.from_numpy(label_weight),
        }

def build_datasets(root=DEFAULT_ROOT, num_slices=24, image_size=224):
    metadata = load_metadata(root)
    train_df, train_series_df, test_df, test_series_df, sample_submission = metadata
    train_dataset = KneeDataset(train_df, train_series_df, num_slices, image_size)
    test_dataset = KneeDataset(test_df, test_series_df, num_slices, image_size)
    return train_dataset, test_dataset, sample_submission, metadata


def build_dataloaders(root=DEFAULT_ROOT, batch_size=2, num_workers=4, num_slices=24, image_size=224):
    train_dataset, test_dataset, sample_submission, metadata = build_datasets(root, num_slices, image_size)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader, sample_submission, metadata


if __name__ == "__main__":
    train_dataset, test_dataset, _, _ = build_datasets()
    sample = train_dataset[0]
    print("train studies:", len(train_dataset), "test studies:", len(test_dataset))
    for key, value in sample.items():
        print(key, value.shape if isinstance(value, torch.Tensor) else value)


__all__ = ["DEFAULT_ROOT", "LABELS", "SLOTS", "SERIES_FEATURES", "find_data_root", "load_metadata", "get_sorted_dicom_info", "sort_dicom_files", "read_dicom", "load_series", "KneeDataset", "build_datasets", "build_dataloaders"]


In [ ]:
"""DINOv2-based hierarchical model for RSNA Knee."""

from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModel



def load_backbone(model_dir):
    """Load the Kaggle-attached Hugging Face DINOv2 model without network access."""
    model_dir = Path(model_dir)
    required = [model_dir / "config.json", model_dir / "pytorch_model.bin"]
    missing = [str(path.name) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"DINOv2 Hugging Face model directory is incomplete: {model_dir}; missing {missing}"
        )
    return AutoModel.from_pretrained(str(model_dir), local_files_only=True)


class RSNADINOv2(nn.Module):
    """2.5D DINOv2 encoder + slice aggregation + label-specific slot fusion."""

    def __init__(
        self,
        model_dir=None,
        hidden_dim=256,
        num_slices=24,
        num_heads=8,
        slice_layers=2,
        slot_layers=1,
        dropout=0.1,
        freeze_backbone=True,
        encoder_chunk_size=24,
    ):
        super().__init__()
        self.backbone = load_backbone(model_dir)

        self.num_slices = num_slices
        self.encoder_chunk_size = encoder_chunk_size
        self.freeze_backbone(freeze_backbone)
        backbone_dim = self.backbone.config.hidden_size

        self.register_buffer("image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.slice_projection = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, hidden_dim))
        self.slice_position = nn.Parameter(torch.zeros(1, num_slices, hidden_dim))

        slice_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slice_encoder = nn.TransformerEncoder(slice_layer, slice_layers, enable_nested_tensor=False)
        self.slice_attention = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 1))

        self.metadata_projection = nn.Sequential(
            nn.Linear(len(SERIES_FEATURES), hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim),
        )
        self.slot_embedding = nn.Parameter(torch.zeros(1, len(SLOTS), hidden_dim))
        slot_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slot_encoder = nn.TransformerEncoder(slot_layer, slot_layers, enable_nested_tensor=False)

        self.label_queries = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.label_weight = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_bias = nn.Parameter(torch.zeros(len(LABELS)))
        nn.init.trunc_normal_(self.slice_position, std=0.02)
        nn.init.trunc_normal_(self.slot_embedding, std=0.02)
        nn.init.trunc_normal_(self.label_queries, std=0.02)
        nn.init.trunc_normal_(self.label_weight, std=0.02)

    def freeze_backbone(self, freeze=True):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze

    def unfreeze_last_blocks(self, num_blocks=2):
        self.freeze_backbone(True)
        for block in self.backbone.encoder.layer[-num_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in self.backbone.layernorm.parameters():
            parameter.requires_grad = True

    @staticmethod
    def make_2p5d(images):
        """把前一张、当前张、后一张切片组成 DINOv2 的三个输入通道。"""
        gray = images.squeeze(3)
        previous = torch.cat([gray[:, :, :1], gray[:, :, :-1]], dim=2)
        following = torch.cat([gray[:, :, 1:], gray[:, :, -1:]], dim=2)
        return torch.stack([previous, gray, following], dim=3)

    def encode_images(self, images, slot_mask):
        batch, slots, slices, _, height, width = images.shape
        images = self.make_2p5d(images).reshape(batch * slots * slices, 3, height, width)
        valid = slot_mask[:, :, None].expand(batch, slots, slices).reshape(-1).bool()
        valid_indices = valid.nonzero(as_tuple=False).squeeze(1)
        images = images[valid_indices]
        images = (images - self.image_mean) / self.image_std

        use_grad = any(parameter.requires_grad for parameter in self.backbone.parameters())
        context = nullcontext() if use_grad else torch.no_grad()
        with context:
            chunks = [self.backbone(pixel_values=chunk).last_hidden_state[:, 0] for chunk in images.split(self.encoder_chunk_size)]
        encoded = torch.cat(chunks)
        all_encoded = encoded.new_zeros(batch * slots * slices, encoded.shape[-1])
        all_encoded = all_encoded.index_copy(0, valid_indices, encoded)
        return all_encoded.view(batch, slots, slices, -1)

    def forward(self, images, slot_mask, series_features=None):
        batch, slots, slices = images.shape[:3]
        if slices != self.num_slices:
            raise ValueError(f"Expected {self.num_slices} slices, got {slices}")

        features = self.encode_images(images, slot_mask)
        features = self.slice_projection(features).view(batch * slots, slices, -1)
        features = self.slice_encoder(features + self.slice_position)
        attention = self.slice_attention(features).softmax(dim=1)
        slot_features = (features * attention).sum(dim=1).view(batch, slots, -1)

        if series_features is not None:
            slot_features = slot_features + self.metadata_projection(series_features)
        slot_features = self.slot_encoder(slot_features + self.slot_embedding, src_key_padding_mask=~slot_mask.bool())

        queries = self.label_queries.unsqueeze(0).expand(batch, -1, -1)
        label_features, _ = self.label_attention(queries, slot_features, slot_features, key_padding_mask=~slot_mask.bool())
        label_features = self.output_norm(label_features)
        return (label_features * self.label_weight.unsqueeze(0)).sum(dim=-1) + self.label_bias


__all__ = ["RSNADINOv2"]


In [ ]:
"""Train RSNA Knee DINOv2 model with single GPU or DistributedDataParallel."""

import atexit
import argparse
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler



def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-root", type=Path, default=DEFAULT_ROOT, help="RSNA data directory; auto-detected under /kaggle/input when omitted")
    parser.add_argument("--labels-csv", type=Path, default=None, help="Soft-label CSV; auto-detected under /kaggle/input when omitted")
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/rsna_outputs"))
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--batch-size", type=int, default=1, help="每张GPU的batch size")
    parser.add_argument("--accum-steps", type=int, default=2)
    parser.add_argument("--num-workers", type=int, default=4, help="每个DDP进程的worker数量")
    parser.add_argument("--num-slices", type=int, default=32)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--dinov2-model-dir", type=Path, default=None, help="Attached Hugging Face DINOv2 directory with config.json and pytorch_model.bin; auto-detected when omitted")
    parser.add_argument("--backbone-mode", choices=["frozen", "last2", "last4", "full"], default="frozen")
    parser.add_argument("--head-lr", type=float, default=3e-4)
    parser.add_argument("--backbone-lr", type=float, default=1e-5)
    parser.add_argument("--weight-decay", type=float, default=0.05)
    parser.add_argument("--warmup-ratio", type=float, default=0.1)
    # T4 (Turing) has fast FP16 Tensor Cores but no native BF16 support.
    parser.add_argument("--amp", choices=["bf16", "fp16", "none"], default="fp16")
    parser.add_argument("--encoder-chunk-size", type=int, default=24)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--no-pos-weight", action="store_true")
    parser.add_argument("--init-checkpoint", type=Path, default=None, help="只加载模型权重，用于冻结阶段后微调")
    parser.add_argument("--resume", type=Path, default=None, help="恢复同配置训练，包括优化器和调度器")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--log-interval", type=int, default=20)
    return parser.parse_args()


def setup_distributed():
    distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if distributed:
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        dist.init_process_group("nccl", device_id=torch.device(f"cuda:{local_rank}"))
        atexit.register(cleanup_distributed)
        return True, local_rank, dist.get_rank(), dist.get_world_size()
    return False, 0, 0, 1


def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def set_seed(seed, rank=0):
    seed += rank
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_label_column(series):
    text = series.astype(str).str.strip().str.upper()
    mapped = text.map({"YES": 1.0, "Y": 1.0, "TRUE": 1.0, "NO": 0.0, "N": 0.0, "FALSE": 0.0, "UNK": 0.5, "UNKNOWN": 0.5, "NAN": np.nan, "": np.nan})
    numeric = pd.to_numeric(series, errors="coerce")
    values = mapped.where(mapped.notna(), numeric)
    return values.where(values.between(0.0, 1.0), np.nan).astype(np.float32)


def resolve_labels_csv(path, data_root):
    if path is not None:
        if path.is_file():
            return path.resolve()
        raise FileNotFoundError(f"Soft-label CSV not found: {path}")
    candidates = list(Path("/kaggle/input").rglob("llm_labels_v4_blend.csv"))
    candidates += [candidate for candidate in Path("/kaggle/input").rglob("*.csv") if "label" in candidate.name.lower()]
    usable = []
    for candidate in candidates:
        try:
            if {"StudyInstanceUID", *LABELS}.issubset(pd.read_csv(candidate, nrows=1).columns):
                usable.append(candidate)
        except (OSError, pd.errors.ParserError, UnicodeDecodeError):
            pass
    return one_path(usable, "soft-label CSV", "--labels-csv")


def one_path(candidates, description, flag):
    candidates = sorted({Path(path).resolve() for path in candidates})
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(f"Could not find {description} under /kaggle/input; pass {flag} explicitly.")
    raise RuntimeError(f"Multiple {description} candidates found: {candidates}. Pass {flag} explicitly.")


def resolve_dinov2_model_dir(model_dir):
    if model_dir is not None:
        model_dir = model_dir.resolve()
        candidates = [model_dir]
    else:
        candidates = [path.parent for path in Path("/kaggle/input").rglob("config.json")]
    candidates = [path for path in candidates if (path / "config.json").is_file() and (path / "pytorch_model.bin").is_file()]
    return one_path(candidates, "Hugging Face DINOv2 model directory", "--dinov2-model-dir")


def prepare_studies(train_df, labels_csv):
    labels_df = pd.read_csv(labels_csv)
    missing = [column for column in ["StudyInstanceUID", *LABELS] if column not in labels_df.columns]
    if missing:
        raise ValueError(f"Weak-label CSV missing columns: {missing}")
    studies = train_df[["StudyInstanceUID", *LABELS]].rename(columns={label: f"gold__{label}" for label in LABELS})
    studies = studies.merge(labels_df[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID", how="inner")
    for label in LABELS:
        studies[label] = parse_label_column(studies[label])
        gold_column = f"gold__{label}"
        studies[gold_column] = pd.to_numeric(studies[gold_column], errors="coerce").astype(np.float32)
        has_gold = studies[gold_column].notna()
        studies.loc[has_gold, label] = studies.loc[has_gold, gold_column]
    return studies[studies[LABELS].notna().any(axis=1)].reset_index(drop=True)

def distributed_metadata(root, distributed, rank):
    objects = [load_metadata(root) if rank == 0 else None]
    if distributed:
        dist.broadcast_object_list(objects, src=0)
    return objects[0]


def make_loader(dataset, batch_size, workers, sampler, shuffle, drop_last):
    return DataLoader(
        dataset, batch_size=batch_size, sampler=sampler, shuffle=shuffle if sampler is None else False,
        num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
        prefetch_factor=2 if workers > 0 else None, drop_last=drop_last,
    )


def build_model(args, distributed, rank):
    kwargs = dict(
        model_dir=str(args.dinov2_model_dir),
        num_slices=args.num_slices, freeze_backbone=True,
        encoder_chunk_size=args.encoder_chunk_size,
    )
    if distributed and rank != 0:
        dist.barrier()
    model = RSNADINOv2(**kwargs)
    if distributed and rank == 0:
        dist.barrier()

    if args.backbone_mode == "last2":
        model.unfreeze_last_blocks(2)
    elif args.backbone_mode == "last4":
        model.unfreeze_last_blocks(4)
    elif args.backbone_mode == "full":
        model.freeze_backbone(False)
    if args.init_checkpoint:
        checkpoint = torch.load(args.init_checkpoint, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint.get("model", checkpoint))
    return model


def make_optimizer(model, args):
    backbone_ids = {id(parameter) for parameter in model.backbone.parameters()}
    backbone = [parameter for parameter in model.parameters() if id(parameter) in backbone_ids and parameter.requires_grad]
    head = [parameter for parameter in model.parameters() if id(parameter) not in backbone_ids and parameter.requires_grad]
    groups = [{"params": head, "lr": args.head_lr}]
    if backbone:
        groups.append({"params": backbone, "lr": args.backbone_lr})
    return AdamW(groups, weight_decay=args.weight_decay)


def make_scheduler(optimizer, total_steps, warmup_ratio):
    warmup_steps = int(total_steps * warmup_ratio)

    def schedule(step):
        if warmup_steps and step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1)))

    return LambdaLR(optimizer, schedule)


def calculate_pos_weight(studies):
    values = studies[LABELS].to_numpy(dtype=np.float32)
    confidence = np.where(np.isnan(values), 0, 2 * np.abs(values - 0.5))
    binary = np.nan_to_num(values, nan=0.5) > 0.5
    positives = np.sum(confidence * binary, axis=0)
    negatives = np.sum(confidence * ~binary, axis=0)
    return torch.tensor(np.clip(negatives / np.maximum(positives, 1), 1, 10), dtype=torch.float32)


def masked_bce(logits, targets, mask, label_weight, pos_weight):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=pos_weight)
    weight = mask * label_weight
    return (loss * weight).sum() / weight.sum().clamp_min(1)


def move_batch(batch, device):
    return {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value for key, value in batch.items()}


def autocast_context(args, device):
    enabled = args.amp != "none" and device.type == "cuda"
    dtype = torch.bfloat16 if args.amp == "bf16" else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=enabled)


def train_epoch(model, loader, sampler, optimizer, scheduler, scaler, pos_weight, device, args, epoch, rank):
    model.train()
    if sampler is not None:
        sampler.set_epoch(epoch)
    optimizer.zero_grad(set_to_none=True)
    total_loss, steps = 0.0, 0

    for step, batch in enumerate(loader):
        batch = move_batch(batch, device)
        with autocast_context(args, device):
            logits = model(batch["images"], batch["slot_mask"], batch["series_features"])
            loss = masked_bce(logits, batch["targets"], batch["label_mask"], batch["label_weight"], pos_weight)
            scaled_loss = loss / args.accum_steps
        scaler.scale(scaled_loss).backward()

        update = (step + 1) % args.accum_steps == 0 or step + 1 == len(loader)
        if update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        total_loss += loss.detach().item()
        steps += 1
        if rank == 0 and (step + 1) % args.log_interval == 0:
            print(f"epoch={epoch + 1} step={step + 1}/{len(loader)} loss={total_loss / steps:.5f}", flush=True)

    stats = torch.tensor([total_loss, steps], device=device, dtype=torch.float64)
    if dist.is_initialized():
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
    return (stats[0] / stats[1]).item()


@torch.no_grad()
def validate(model, loader, device, args, distributed, world_size):
    model.eval()
    local = {"uid": [], "pred": [], "target": [], "mask": [], "weight": []}
    for batch in loader:
        batch = move_batch(batch, device)
        with autocast_context(args, device):
            logits = model(batch["images"], batch["slot_mask"], batch["series_features"])
        local["uid"].extend(batch["study_uid"])
        local["pred"].append(torch.sigmoid(logits).float().cpu().numpy())
        local["target"].append(batch["targets"].float().cpu().numpy())
        local["mask"].append(batch["label_mask"].float().cpu().numpy())
        local["weight"].append(batch["label_weight"].float().cpu().numpy())
    for key in ["pred", "target", "mask", "weight"]:
        local[key] = np.concatenate(local[key])

    gathered = [None] * world_size if distributed else [local]
    if distributed:
        dist.all_gather_object(gathered, local)

    merged = {}
    for part in gathered:
        for index, uid in enumerate(part["uid"]):
            merged[uid] = (part["pred"][index], part["target"][index], part["mask"][index], part["weight"][index])
    uids = list(merged)
    predictions = np.stack([value[0] for value in merged.values()])
    targets = np.stack([value[1] for value in merged.values()])
    masks = np.stack([value[2] for value in merged.values()])
    weights = np.stack([value[3] for value in merged.values()])

    aucs = {}
    for index, label in enumerate(LABELS):
        valid = (masks[:, index] > 0) & (weights[:, index] > 0)
        y_true, y_pred = (targets[valid, index] > 0.5).astype(np.int64), predictions[valid, index]
        aucs[label] = float(roc_auc_score(y_true, y_pred)) if len(np.unique(y_true)) == 2 else float("nan")
    macro_auc = float(np.nanmean(list(aucs.values())))

    return macro_auc, aucs

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_auc, args):
    raw_model = model.module if isinstance(model, DDP) else model
    torch.save({
        "model": raw_model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
        "epoch": epoch, "best_auc": best_auc, "args": vars(args),
    }, path)


def main():
    args = parse_args()
    distributed, local_rank, rank, world_size = setup_distributed()
    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")
    set_seed(args.seed, rank)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    args.data_root = find_data_root(args.data_root)
    metadata = distributed_metadata(args.data_root, distributed, rank)
    train_df, train_series_df = metadata[0], metadata[1]
    args.labels_csv = resolve_labels_csv(args.labels_csv, args.data_root)
    args.dinov2_model_dir = resolve_dinov2_model_dir(args.dinov2_model_dir)
    studies = prepare_studies(train_df, args.labels_csv)
    gold_columns = [f"gold__{label}" for label in LABELS]
    is_gold = studies[gold_columns].notna().all(axis=1)
    train_studies = studies[~is_gold].reset_index(drop=True)
    valid_studies = studies[is_gold].reset_index(drop=True)
    for label in LABELS:
        valid_studies[label] = valid_studies[f"gold__{label}"].astype(np.float32)
    if len(valid_studies) != 58:
        raise ValueError(f"Expected 58 fully labeled validation studies, found {len(valid_studies)}")
    if rank == 0:
        print(f"labels={args.labels_csv}")
        print(f"DINOv2 Hugging Face model={args.dinov2_model_dir}")
        print(f"split: weak-label train={len(train_studies)} | gold-label valid={len(valid_studies)} | total={len(studies)}")

    train_dataset = KneeDataset(train_studies, train_series_df, args.num_slices, args.image_size)
    valid_dataset = KneeDataset(valid_studies, train_series_df, args.num_slices, args.image_size)
    train_sampler = DistributedSampler(train_dataset, shuffle=True, drop_last=False) if distributed else None
    valid_sampler = DistributedSampler(valid_dataset, shuffle=False, drop_last=False) if distributed else None
    train_loader = make_loader(train_dataset, args.batch_size, args.num_workers, train_sampler, True, False)
    valid_loader = make_loader(valid_dataset, args.batch_size, args.num_workers, valid_sampler, False, False)

    model = build_model(args, distributed, rank).to(device)
    resume_checkpoint = torch.load(args.resume, map_location="cpu", weights_only=False) if args.resume else None
    if resume_checkpoint:
        model.load_state_dict(resume_checkpoint["model"])
    if distributed:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    raw_model = model.module if isinstance(model, DDP) else model
    optimizer = make_optimizer(raw_model, args)
    updates_per_epoch = math.ceil(len(train_loader) / args.accum_steps)
    scheduler = make_scheduler(optimizer, args.epochs * updates_per_epoch, args.warmup_ratio)
    scaler = torch.amp.GradScaler("cuda", enabled=args.amp == "fp16" and device.type == "cuda")
    start_epoch, best_auc = 0, -float("inf")
    if resume_checkpoint:
        optimizer.load_state_dict(resume_checkpoint["optimizer"])
        scheduler.load_state_dict(resume_checkpoint["scheduler"])
        scaler.load_state_dict(resume_checkpoint["scaler"])
        start_epoch, best_auc = resume_checkpoint["epoch"] + 1, resume_checkpoint["best_auc"]

    pos_weight = calculate_pos_weight(train_studies).to(device) if not args.no_pos_weight else torch.ones(len(LABELS), device=device)
    output_dir = args.output_dir
    if rank == 0:
        output_dir.mkdir(parents=True, exist_ok=True)
        pd.concat([
            train_studies[["StudyInstanceUID"]].assign(split="train"),
            valid_studies[["StudyInstanceUID"]].assign(split="valid"),
        ], ignore_index=True).to_csv(output_dir / "split.csv", index=False)
        print("trainable parameters:", sum(parameter.numel() for parameter in raw_model.parameters() if parameter.requires_grad))

    for epoch in range(start_epoch, args.epochs):
        train_loss = train_epoch(model, train_loader, train_sampler, optimizer, scheduler, scaler, pos_weight, device, args, epoch, rank)
        macro_auc, aucs = validate(model, valid_loader, device, args, distributed, world_size)
        if rank == 0:
            print(f"epoch={epoch + 1} train_loss={train_loss:.5f} val_macro_auc={macro_auc:.5f}")
            print(json.dumps(aucs, ensure_ascii=False, indent=2))
            improved = macro_auc > best_auc
            best_auc = max(best_auc, macro_auc)
            save_checkpoint(output_dir / "last.pt", model, optimizer, scheduler, scaler, epoch, best_auc, args)
            if improved:
                save_checkpoint(output_dir / "best.pt", model, optimizer, scheduler, scaler, epoch, best_auc, args)
        if distributed:
            dist.barrier()

    cleanup_distributed()


# This notebook cell itself starts two forked workers, one per T4 GPU.
# No .py file is created or executed.
def _notebook_ddp_worker(local_rank, world_size):
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29500"
    os.environ["RANK"] = str(local_rank)
    os.environ["LOCAL_RANK"] = str(local_rank)
    os.environ["WORLD_SIZE"] = str(world_size)
    import sys
    sys.argv = ["notebook_train"]  # ignore Kaggle/IPython command-line arguments
    main()


if torch.cuda.device_count() < 2:
    raise RuntimeError("Enable the Kaggle accelerator with 2 × T4 GPUs before running this cell.")
torch.multiprocessing.start_processes(
    _notebook_ddp_worker, args=(2,), nprocs=2, join=True, start_method="fork"
)
